In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 뉴스 정보원 자동 추출 시스템 - LEAN VERSION
# EXAONE 4.0.1 Non-reasoning 모드 최적화
# RAG를 붙일지 고민해봐야겠는데.;
# ═══════════════════════════════════════════════════════════════════════

import pandas as pd
import json
import re
import os
from tqdm import tqdm
import openai

# ═══════════════════════════════════════════════════════════════════════
# 1. 정보원 유형 정의
# ═══════════════════════════════════════════════════════════════════════

SOURCE_TYPES = {
    "대통령" : "대한민국 대통령",
    "국회의원": "국회의원/정치인",
    "장관": "정부부처 장관",
    "실명": "실명이 명시된 개인",
    "익명": "익명 정보원",
    "기관": "정부/공공기관",
    "정당": "정당/정치조직",
    "시민단체": "NGO/시민단체",
    "언론사": "언론매체",
    "기업": "민간기업",
    "일반시민": "일반시민",
    "유튜버": "유튜브 채널"
    "기타": "기타"
}

def format_source_types():
    """정보원 유형 목록 문자열 생성"""
    return "\n".join([f"- {k}: {v}" for k, v in SOURCE_TYPES.items()])

# ═══════════════════════════════════════════════════════════════════════
# 2. 프롬프트 (JSON 예시 추가)
# ═══════════════════════════════════════════════════════════════════════

SOURCE_EXTRACTION_PROMPT = """당신은 한국 뉴스 정보원 추출 전문가입니다.

**[작업]**
1. 정보원을 모두 추출하세요 (사람은 이름만)
2. 각 정보원의 유형을 분류하세요
3. 신뢰도를 평가하세요 (0-100)

**[정보원 유형]**
{source_types}

**[예시]**
입력: "환경부는 발표했다. 그린피스는 비판했다."
출력:
{{
  "source": ["환경부", "그린피스"],
  "source_type": ["기관", "시민단체"],
  "confidence": 98,
  "reasoning": "두 정보원 명확함"
}}

**[규칙]**
- source와 source_type 길이 동일
- 중복 정보원 제거
- JSON만 출력 (설명 금지)
- <think> 태그 사용 금지

**[뉴스 텍스트]**
{text}

**[JSON 응답]**
{{
  "source": ["정보원1", "정보원2"],
  "source_type": ["유형1", "유형2"],
  "confidence": 95,
  "reasoning": "분석근거"
}}"""

# ═══════════════════════════════════════════════════════════════════════
# 3. 유틸리티 함수 (최소화)
# ═══════════════════════════════════════════════════════════════════════

def clean_text(text):
    """텍스트 정제 (제어문자 제거 강화)"""
    if pd.isna(text) or text == "":
        return "텍스트 없음"

    text = str(text)

    # 제어문자 완전 제거
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = text.replace('\t', ' ')
    text = text.replace('\b', ' ')
    text = text.replace('\f', ' ')
    text = text.replace('\v', ' ')

    # /m 같은 특수 시퀀스 제거
    text = re.sub(r'/[a-z]', '', text)

    # 모든 제어문자 제거
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', text)

    # 연속 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()

    return text if text else "텍스트 없음"

def parse_json(response):
    """JSON 파싱 (간소화)"""
    try:
        # 제어문자 제거
        response = clean_text(response)

        # Markdown 코드 블록 제거
        response = re.sub(r'```json\s*', '', response)
        response = re.sub(r'```\s*', '', response)

        # JSON 추출
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return None
    except:
        return None

# ═══════════════════════════════════════════════════════════════════════
# 4. 정보원 분석기 (LEAN VERSION)
# ═══════════════════════════════════════════════════════════════════════

class SourceAnalyzer:
    """초경량 정보원 분석기"""

    def __init__(self, client, output_dir='./output/'):
        self.client = client
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def analyze(self, text):
        """단일 텍스트 분석 (핵심 로직만)"""
        text = clean_text(text)
        prompt = SOURCE_EXTRACTION_PROMPT.format(
            text=text,
            source_types=format_source_types()
        )

        try:
            response = self.client.chat.completions.create(
                model="lgai-exaone_exaone-4.0.1-32b",
                messages=[
                    {
                        "role": "system",
                        "content": "당신은 뉴스 기사의 정보원 추출 전문가입니다. JSON 형식으로만 정확히 출력하세요."
                    },
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                max_tokens=512,
                extra_body={
                    "chat_template_kwargs": {
                        "enable_thinking": False
                    }
                }
            )

            result = parse_json(response.choices[0].message.content)

            if result and 'source' in result and 'source_type' in result:
                return result
            else:
                return {
                    "source": ["파싱실패"],
                    "source_type": ["기타"],
                    "confidence": 0,
                    "reasoning": "JSON 파싱 실패"
                }

        except Exception as e:
            return {
                "source": ["오류"],
                "source_type": ["기타"],
                "confidence": 0,
                "reasoning": f"API 오류: {str(e)}"
            }

    def process_dataframe(self, df, text_column='text', output_file='results.csv'):
        """데이터프레임 배치 처리"""
        output_path = os.path.join(self.output_dir, output_file)
        checkpoint_path = os.path.join(self.output_dir, f"checkpoint_{output_file}")

        # 결과 저장용
        sources = []
        source_types = []
        confidences = []
        reasonings = []

        # 체크포인트 복구
        start_idx = 0
        if os.path.exists(checkpoint_path):
            try:
                cp_df = pd.read_csv(checkpoint_path, encoding='utf-8-sig')
                if 'source' in cp_df.columns:
                    completed = sum(~cp_df['source'].isna())
                    if completed > 0:
                        start_idx = completed
                        sources = cp_df['source'].tolist()[:completed]
                        source_types = cp_df['source_type'].tolist()[:completed]
                        confidences = cp_df['confidence'].tolist()[:completed]
                        reasonings = cp_df['reasoning'].tolist()[:completed]
                        print(f"📂 체크포인트 복구: {start_idx}행부터 재개")
            except:
                pass

        # 진행 상황 표시
        for i in tqdm(range(start_idx, len(df)), desc="분석 중"):
            text = df.iloc[i][text_column]

            if pd.isna(text):
                text = "텍스트 없음"

            result = self.analyze(text)

            sources.append(", ".join(result['source']))
            source_types.append(", ".join(result['source_type']))
            confidences.append(result['confidence'])
            reasonings.append(result.get('reasoning', ''))

            # 체크포인트 (50개마다)
            if (i + 1) % 50 == 0:
                temp_df = df.copy()
                temp_df['source'] = sources + [''] * (len(temp_df) - len(sources))
                temp_df['source_type'] = source_types + [''] * (len(temp_df) - len(source_types))
                temp_df['confidence'] = confidences + [0] * (len(temp_df) - len(confidences))
                temp_df['reasoning'] = reasonings + [''] * (len(temp_df) - len(reasonings))
                temp_df.to_csv(checkpoint_path, encoding='utf-8-sig', index=False)

        # 최종 저장
        result_df = df.copy()
        result_df['source'] = sources
        result_df['source_type'] = source_types
        result_df['confidence'] = confidences
        result_df['reasoning'] = reasonings

        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)

        # 간단한 통계
        print(f"\n완료: {len(result_df)}건")
        print(f"평균 신뢰도: {result_df['confidence'].mean():.1f}%")
        print(f"저장: {output_path}")

        return result_df

# ═══════════════════════════════════════════════════════════════════════
# 5. 메인 실행
# ═══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # LM Studio 연결
    client = openai.OpenAI(
        base_url="http://218.156.182.231:1234/v1",
        api_key="lm-studio"
    )

    # 분석기 초기화
    analyzer = SourceAnalyzer(
        client=client,
        output_dir='./output/'
    )

    # CSV 로드
    input_file = "/mnt/c/Users/com/Desktop/프로젝트/5.18뉴스+댓글데이터_241201-250515/1. data/202508-202509/merged_output_news_cleaned001.csv"  # ⚠️ 경로 수정

    if os.path.exists(input_file):
        df = pd.read_csv(input_file, encoding='utf-8-sig')
        print(f"데이터 로드: {len(df)}개 행\n")

        # 분석 실행
        result_df = analyzer.process_dataframe(
            df=df,
            text_column='text',
            output_file='/mnt/c/Users/com/Desktop/프로젝트/5.18뉴스+댓글데이터_241201-250515/1. data/202508-202509/merged_output_news_cleaned002.csv'
        )
    else:
        print(f"파일 없음: {input_file}")

In [ ]:
{
  "type": "object",
  "properties": {
    "source": {
      "type": "array",
      "items": {
        "type": "string",
        "minLength": 1,
        "maxLength": 100
      },
      "minItems": 0,
      "maxItems": 20,
      "description": "뉴스 기사에서 추출된 정보원 목록"
    },
    "source_type": {
      "type": "array",
      "items": {
        "type": "string",
        "enum": [
          "실명",
          "익명",
          "기관",
          "정당",
          "시민단체",
          "언론사",
          "기업",
          "일반시민",
          "기타"
        ]
      },
      "minItems": 0,
      "maxItems": 20,
      "description": "각 정보원의 유형 분류"
    },
    "confidence": {
      "type": "integer",
      "minimum": 0,
      "maximum": 100,
      "description": "분석 결과의 신뢰도"
    },
    "reasoning": {
      "type": "string",
      "minLength": 10,
      "maxLength": 500,
      "description": "분석 근거 (최소 10자, 최대 500자)"
    }
  },
  "required": [
    "source",
    "source_type",
    "confidence",
    "reasoning"
  ],
  "additionalProperties": false
}